# Customer Churn Prediction (Telecom)
**Goal:** predict whether a telecom customer will leave (churn) using the IBM Telco dataset.

**Pipeline:** Load data → EDA → Preprocessing → Train 3 models → Evaluate → Save the best model for the Streamlit app.

Run every cell from top to bottom (Runtime → Run all).

In [ ]:
!pip install -q xgboost

## 1. Imports

In [ ]:
import os, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (accuracy_score, roc_auc_score, recall_score, precision_score,
                             f1_score, classification_report, confusion_matrix,
                             ConfusionMatrixDisplay, roc_curve)

warnings.filterwarnings("ignore")
os.makedirs("assets", exist_ok=True)   # charts are saved here for the README
sns.set_theme(style="whitegrid")

## 2. Load the dataset (IBM Telco Customer Churn)

In [ ]:
URL = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(URL)
print("Rows, columns:", df.shape)
df.head()

## 3. Exploratory Data Analysis (EDA)

In [ ]:
print(df['Churn'].value_counts())
print((df['Churn'].value_counts(normalize=True) * 100).round(1))

plt.figure(figsize=(5, 4))
sns.countplot(x='Churn', data=df, palette='Set2')
plt.title('Churn distribution')
plt.tight_layout()
plt.savefig('assets/01_churn_distribution.png', dpi=150)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))

sns.countplot(x='Contract', hue='Churn', data=df, palette='Set2', ax=axes[0])
axes[0].set_title('Contract type vs churn')

sns.boxplot(x='Churn', y='MonthlyCharges', data=df, palette='Set2', ax=axes[1])
axes[1].set_title('Monthly charges vs churn')

sns.histplot(data=df, x='tenure', hue='Churn', bins=30, kde=True, ax=axes[2])
axes[2].set_title('Tenure vs churn')

plt.tight_layout()
plt.savefig('assets/02_eda_overview.png', dpi=150)
plt.show()

In [ ]:
# Churn rate for each contract type (useful business insight)
churn_by_contract = df.groupby('Contract')['Churn'].apply(lambda s: (s == 'Yes').mean() * 100).round(1)
print("Churn rate % by contract:")
print(churn_by_contract)

## 4. Preprocessing
- `TotalCharges` looks numeric but has blank strings → convert and drop those rows
- Drop `customerID` (just an identifier)
- Encode text columns to numbers (and **save the encodings** so the Streamlit app uses the exact same mapping)

In [ ]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
print("Rows with blank TotalCharges:", df['TotalCharges'].isnull().sum())
df = df.dropna().reset_index(drop=True)
df = df.drop(columns=['customerID'])
df['SeniorCitizen'] = df['SeniorCitizen'].map({0: 'No', 1: 'Yes'})

encoders = {}
for col in df.select_dtypes(include='object').columns:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    encoders[col] = list(le.classes_)      # position in this list = encoded number

print("Encodings learned:")
for k, v in encoders.items():
    print(f"  {k}: {v}")
print("Shape after cleaning:", df.shape)

In [ ]:
X = df.drop(columns=['Churn'])
y = df['Churn']          # 1 = churned, 0 = stayed

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print("Train:", X_train.shape, "| Test:", X_test.shape)

## 5. Train and compare 3 models

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=2000),
    "Random Forest": RandomForestClassifier(n_estimators=300, max_depth=8, random_state=42),
    "XGBoost": XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=3,
                             subsample=0.9, colsample_bytree=0.9,
                             eval_metric='logloss', random_state=42),
}

rows = []
for name, m in models.items():
    m.fit(X_train, y_train)
    pred = m.predict(X_test)
    proba = m.predict_proba(X_test)[:, 1]
    rows.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred),
        "Recall": recall_score(y_test, pred),
        "F1": f1_score(y_test, pred),
        "ROC-AUC": roc_auc_score(y_test, proba),
    })

results = pd.DataFrame(rows).set_index("Model").round(3)
results

> **Reading the table:** only ~27% of customers churn, so *accuracy alone is misleading*.
> **Recall** = how many real churners we catch. **ROC-AUC** = how well the model ranks churners above non-churners.

## 6. Evaluate the XGBoost model

In [ ]:
best_model = models["XGBoost"]
pred = best_model.predict(X_test)
proba = best_model.predict_proba(X_test)[:, 1]
print(classification_report(y_test, pred, target_names=['Stayed', 'Churned']))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

ConfusionMatrixDisplay(confusion_matrix(y_test, pred),
                       display_labels=['Stayed', 'Churned']).plot(ax=axes[0], cmap='Blues', colorbar=False)
axes[0].set_title('Confusion matrix')

fpr, tpr, _ = roc_curve(y_test, proba)
axes[1].plot(fpr, tpr, label=f"AUC = {roc_auc_score(y_test, proba):.3f}")
axes[1].plot([0, 1], [0, 1], 'k--', label='Random guess')
axes[1].set_xlabel('False positive rate'); axes[1].set_ylabel('True positive rate')
axes[1].set_title('ROC curve'); axes[1].legend()

imp = pd.Series(best_model.feature_importances_, index=X.columns).sort_values().tail(10)
imp.plot(kind='barh', ax=axes[2], color='#1D9E75')
axes[2].set_title('Top 10 features')

plt.tight_layout()
plt.savefig('assets/03_model_evaluation.png', dpi=150)
plt.show()

## 7. Save everything the Streamlit app needs

In [ ]:
best_model.save_model("churn_model.json")        # XGBoost's own portable format
with open("columns.json", "w") as f:
    json.dump(list(X.columns), f)                 # exact column order used in training
with open("encoders.json", "w") as f:
    json.dump(encoders, f, indent=2)              # text -> number mappings
results.to_csv("model_results.csv")

print("Saved: churn_model.json, columns.json, encoders.json, model_results.csv")

In [ ]:
# Sanity check: reload the saved model and predict one customer
check = XGBClassifier()
check.load_model("churn_model.json")
print("Churn probability of first test customer:", round(float(check.predict_proba(X_test.iloc[[0]])[0][1]), 3))

In [ ]:
# Colab only: download the files to your computer (skipped automatically elsewhere)
try:
    from google.colab import files
    for fn in ["churn_model.json", "columns.json", "encoders.json"]:
        files.download(fn)
except ImportError:
    pass